# Penalties Analysis

Common setup and helpers are defined once here.

Data: auto-loads every `{chain}_*.csv` in `PENALTIES_DATA_DIR` and groups by `blockchain`. Cross-chain views activate automatically once >1 chain is present.

In [ ]:
# === COMMON SETUP & HELPERS (defined once) ===
import glob, os, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 80)
plt.rcParams.update({'figure.figsize': (9, 5), 'axes.grid': True, 'grid.alpha': 0.3})
DATA_DIR = os.environ.get('PENALTIES_DATA_DIR', '.')
NIL = {'<nil>': np.nan, 'None': np.nan, 'nan': np.nan, '': np.nan}
WEI = 1e18

NUMERIC = ['volume_native','slippage_native','reward_penalty_native','reward_penalty_uncapped_native',
    'reward_native','penalty_native','penalty_uncapped_native','penalty_cap_native','reward_cap_upper_native',
    'reference_score','observed_score','execution_cost_native','slippage_usd','order_size_usd','markout_usd',
    'markout_relative','slippage_tolerance_bps','calculated_slippage_bps','seconds_since_created',
    'seconds_to_settle','auction_id','settlement_block','block_deadline']
BOOL = ['settled','is_excluded_from_penalties','is_quote_reward_eligible','partially_fillable',
    'is_out_of_market','smart_slippage']
# whole-token (/WEI) THEN USD (avoids 1e18 blowup with a per-token price)
ECON_NATIVE = ['reward_native','penalty_native','penalty_uncapped_native','penalty_cap_native',
    'reward_penalty_native','reward_cap_upper_native','execution_cost_native','reference_score','observed_score']

def _num(s): return pd.to_numeric(s.astype(str).replace(NIL), errors='coerce')

def load_chain_csvs(data_dir=DATA_DIR, pattern='*_*_*.csv'):
    """Load + concat every chain CSV; print provenance (name, rows, md5) to pin the extract."""
    paths = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not paths: raise FileNotFoundError(f'no CSVs matching {pattern} in {data_dir!r}')
    frames = []
    for p in paths:
        d = pd.read_csv(p, low_memory=False); d['_source_file'] = os.path.basename(p)
        md5 = hashlib.md5(open(p, 'rb').read()).hexdigest()[:12]
        print(f'  {os.path.basename(p):45s} rows={len(d):>7,}  md5={md5}')
        frames.append(d)
    return pd.concat(frames, ignore_index=True)

def native_token_usd_price(df):
    """Per-chain-day median native-token USD price from settled rows (order_size_usd / volume_tok),
    applied to all rows incl reverts. Stable vs noisy per-row slippage ratios. Falls back to per-chain."""
    s = df['settled'] == True
    vol_tok = df['volume_native'] / WEI
    px = (df['order_size_usd'] / vol_tok).where(s & (vol_tok > 0))
    day = pd.to_datetime(df['auction_timestamp'], errors='coerce').dt.date
    key = pd.DataFrame({'chain': df['blockchain'], 'day': day, 'px': px})
    cd = key.groupby(['chain', 'day'])['px'].transform('median')
    ch = key.groupby('chain')['px'].transform('median')
    return cd.fillna(ch)

def prepare(df):
    """Coerce types, derive native→USD price, build *_tok/*_usd economic cols, two P&L concepts, bps, flags.
    Grains: attempt = (auction_id, order_uid); reward/penalty = (auction_id, solver); markout/size_usd = settled-only."""
    df = df.copy()
    for c in NUMERIC:
        if c in df: df[c] = _num(df[c])
    for c in BOOL:
        if c in df: df[c] = df[c].astype(str).str.lower().map({'true': True, 'false': False})
    df['native_usd_price'] = native_token_usd_price(df)             # USD per whole token
    for c in ECON_NATIVE:                                           # wei -> token -> USD
        base = c[:-len('_native')] if c.endswith('_native') else c
        df[base + '_tok'] = df[c] / WEI
        df[base + '_usd'] = df[base + '_tok'] * df['native_usd_price']
    df['volume_tok'] = df['volume_native'] / WEI
    df['volume_usd'] = df['volume_tok'] * df['native_usd_price']
    # size_usd: Dune order_size_usd is settled-only; fall back to volume_usd (present on reverts)
    df['size_usd'] = df['order_size_usd'].where(df['order_size_usd'].notna(), df['volume_usd'])
    # GROSS excludes reward/penalty. Using when penalty/cap is the regressor (else tautological).
    # NET includes them. Realised welfare under the mechanism.
    df['gross_execution_pnl_usd'] = np.where(df['settled'] == True, df['slippage_usd'] - df['execution_cost_usd'], np.nan)
    df['net_mechanism_pnl_usd'] = (df['reward_penalty_usd'].fillna(0) + df['slippage_usd'].fillna(0) - df['execution_cost_usd'].fillna(0))
    df['reverted'] = ~df['settled'].fillna(False)
    df['cap_binds'] = df['penalty_uncapped_native'] > df['penalty_cap_native']
    df['is_solo_bidder'] = df['reference_score'] == 0              # reference_score == 0 = penalty structurally 0
    df['markout_bps'] = df['markout_relative'] * 1e4
    df['penalty_bps'] = (df['penalty_usd'] / df['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    df['reward_bps'] = (df['reward_usd'] / df['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    return df

def apply_filters(df, fok_only=True, in_market_only=True, exclude_penalty_excluded=True):
    """In-market fill-or-kill only; reverted winners KEPT (the signal). Returns (df, funnel)."""
    funnel = {'start': len(df)}; out = df
    if fok_only: out = out[out['partially_fillable'] == False]; funnel['fill_or_kill'] = len(out)
    if in_market_only: out = out[out['is_out_of_market'] == False]; funnel['in_market'] = len(out)
    if exclude_penalty_excluded: out = out[out['is_excluded_from_penalties'] == False]; funnel['penalty_eligible'] = len(out)
    funnel['reverted_kept'] = int(out['reverted'].sum()); funnel['settled_kept'] = int((~out['reverted']).sum())
    return out.copy(), funnel

def reward_per_auction_solver(df):
    """Aggregate to (chain, auction, solver). Reward/penalty repeat across a multi-order solution's
    rows ('first'); order-level economics (gross pnl, volume) are summed."""
    g = df.groupby(['blockchain', 'auction_id', 'solver'], dropna=False)
    out = g.agg(
        solver_name=('solver_name', 'first'),
        reward_usd=('reward_usd', 'first'),
        penalty_usd=('penalty_usd', 'first'),
        penalty_uncapped_usd=('penalty_uncapped_usd', 'first'),
        reward_penalty_usd=('reward_penalty_usd', 'first'),
        penalty_cap_usd=('penalty_cap_usd', 'first'),
        gross_execution_pnl_usd=('gross_execution_pnl_usd', 'sum'),
        volume_usd=('volume_usd', 'sum'),
        reverted=('reverted', 'any'),
        cap_binds=('cap_binds', 'any'),
        order_count=('order_uid', 'nunique'),
    ).reset_index()
    out['net_mechanism_pnl_usd'] = out['reward_penalty_usd'].fillna(0) + out['gross_execution_pnl_usd'].fillna(0)
    out['penalty_bps'] = (out['penalty_usd'] / out['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    out['reward_bps'] = (out['reward_usd'] / out['volume_usd']).replace([np.inf, -np.inf], np.nan) * 1e4
    return out

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z**2/n; c = p + z**2/(2*n)
    h = z*np.sqrt(p*(1-p)/n + z**2/(4*n**2))
    return (p, (c-h)/d, (c+h)/d)

FIXED_BINS = [0, 100, 1_000, 10_000, 100_000, np.inf]             
FIXED_LABELS = ['<$100', '$100-1k', '$1k-10k', '$10k-100k', '>$100k']

def add_fixed_buckets(df, col='size_usd'):
    df = df.copy(); df['size_fixed'] = pd.cut(df[col], bins=FIXED_BINS, labels=FIXED_LABELS); return df

def add_chain_quantile_buckets(df, value='size_usd', q=5):
    """Per-chain quantile buckets (primary size view). duplicates='drop' + ordinal relabel so tie-heavy
    chains don't crash on fixed labels; ordered categorical so Q10 doesn't sort before Q2."""
    out = df.copy(); out['size_q'] = pd.Series(pd.NA, index=out.index, dtype='object')
    for ch, idx in out.groupby('blockchain').groups.items():
        s = out.loc[idx, value]; valid = s[s.notna() & np.isfinite(s) & (s > 0)]
        if len(valid) < q: continue
        try: binned = pd.qcut(valid, q=q, duplicates='drop')
        except ValueError: continue
        out.loc[valid.index, 'size_q'] = binned.cat.codes.map(lambda c: f'Q{c+1}')
    out['size_q'] = pd.Categorical(out['size_q'], categories=[f'Q{i}' for i in range(1, q+1)], ordered=True)
    return out

def ecdf(ax, series, label):
    x = np.sort(pd.Series(series).dropna().values)
    if len(x): ax.plot(x, np.arange(1, len(x)+1)/len(x), label=label)

def barpos(ax, labels, vals, **kw):
    """Bar with explicit numeric x + string labels (avoids matplotlib categorical-float errors)."""
    x = np.arange(len(labels)); ax.bar(x, np.asarray(vals, float), **kw)
    ax.set_xticks(x); ax.set_xticklabels([str(l) for l in labels])

print('helpers defined')

In [ ]:
print('loading:'); RAW = prepare(load_chain_csvs())
DF, FUNNEL = apply_filters(RAW)
print('\nfunnel:', {k: f'{v:,}' for k, v in FUNNEL.items()})
print('chains:', DF['blockchain'].unique().tolist())
MULTI = DF['blockchain'].nunique() > 1
if not MULTI: print('single chain')

Sanity panel before interpretation: per-chain coverage + solver HHI, null-on-revert check for settled-only columns, and p50/p99/max to catch native-unit / huge-size artifacts that dominate totals.

In [ ]:
def coverage_table(df):
    rows = []
    for ch, d in df.groupby('blockchain'):
        shares = d['solver'].value_counts(normalize=True)
        rows.append({'chain': ch, 'attempts': len(d), 'orders': d['order_uid'].nunique(),
            'solvers': d['solver'].nunique(), 'revert_rate': d['reverted'].mean(),
            'penalty_cap_usd': d['penalty_cap_usd'].median(), 'median_size_usd': d['size_usd'].median(),
            'solver_hhi': float((shares**2).sum()), 'top_solver_share': float(shares.iloc[0]),
            'token_pairs': (d['sell_token'].astype(str)+'/'+d['buy_token'].astype(str)).nunique()})
    return pd.DataFrame(rows).set_index('chain')

def missingness_check(df):
    cols = ['order_size_usd','markout_usd','markout_relative','seconds_to_settle','slippage_usd','execution_cost_native']
    rev, st = df[df['reverted']], df[~df['reverted']]
    return pd.DataFrame({'null_on_revert_%': [100*rev[c].isna().mean() for c in cols],
                         'null_on_settled_%': [100*st[c].isna().mean() for c in cols]}, index=cols).round(1)

def outlier_diag(df):
    cols = ['volume_tok','penalty_uncapped_tok','size_usd','penalty_uncapped_usd']
    return {ch: pd.DataFrame({c: [d[c].quantile(.5), d[c].quantile(.99), d[c].max()] for c in cols},
                             index=['p50','p99','max']) for ch, d in df.groupby('blockchain')}

print('=== coverage ==='); display(coverage_table(DF))
print('=== missingness (settled-only cols ~100% null on revert) ==='); display(missingness_check(DF))
print('=== outliers (p50/p99/max) ===')
for ch, t in outlier_diag(DF).items(): print(ch); display(t)

## 1. Revert rates by chain vs rewards & penalties
Per-chain revert rate (attempt grain, Wilson CI) with median reward/penalty (deduped to auction-solver grain).

In [ ]:
def revert_by_chain(df):
    rps = reward_per_auction_solver(df); rows = []
    for ch, d in df.groupby('blockchain'):
        p, lo, hi = wilson(int(d['reverted'].sum()), len(d))
        r = rps[rps['blockchain'] == ch]
        rows.append({'chain': ch, 'n_attempts': len(d), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi,
            'penalty_cap_usd': d['penalty_cap_usd'].median(), 'med_reward_usd': r['reward_usd'].median(),
            'med_penalty_usd': r['penalty_usd'].median(),
            'med_penalty_on_revert_usd': r.loc[r['reverted'], 'penalty_usd'].median()})
    return pd.DataFrame(rows).set_index('chain')

def plot_revert_by_chain(tbl):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5)); x = np.arange(len(tbl))
    lo = (tbl['revert_rate']-tbl['ci_lo']).clip(lower=0).fillna(0); hi = (tbl['ci_hi']-tbl['revert_rate']).clip(lower=0).fillna(0)
    ax[0].bar(x, tbl['revert_rate']*100, yerr=np.vstack([lo, hi])*100, capsize=4, color='#c0504d')
    ax[0].set_xticks(x); ax[0].set_xticklabels([str(i) for i in tbl.index])
    ax[0].set_ylabel('revert rate (%)'); ax[0].set_title('Revert rate by chain (95% Wilson CI)')
    w = 0.38
    ax[1].bar(x-w/2, tbl['med_reward_usd'], w, label='median reward', color='#4f81bd')
    ax[1].bar(x+w/2, tbl['med_penalty_usd'], w, label='median penalty', color='#c0504d')
    ax[1].set_xticks(x); ax[1].set_xticklabels([str(i) for i in tbl.index])
    ax[1].set_ylabel('USD (auction-solver grain)'); ax[1].set_title('Median reward vs penalty'); ax[1].legend()
    plt.tight_layout(); plt.show()

T2 = revert_by_chain(DF); display(T2); plot_revert_by_chain(T2)

## 2. Revert rate by order size
Primary: per-chain quantile buckets (scale-adaptive), with median/min/max USD per bucket so Q-labels stay economically interpretable. Secondary: fixed-$ buckets for cross-chain comparability. Visual: boundary-free rolling revert rate over log10(size).

In [ ]:
def revert_by_quantile(df):
    d = add_chain_quantile_buckets(df); rows = []
    for (ch, q), g in d.groupby(['blockchain','size_q'], observed=True):
        if len(g) == 0: continue
        p, lo, hi = wilson(int(g['reverted'].sum()), len(g))
        rows.append({'chain': ch, 'size_q': q, 'n': len(g), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi,
            'med_size_usd': g['size_usd'].median(), 'min_size_usd': g['size_usd'].min(), 'max_size_usd': g['size_usd'].max(),
            'med_penalty_bps': g.loc[g['reverted'], 'penalty_bps'].median(),
            'med_gross_pnl_usd': g['gross_execution_pnl_usd'].median()})
    return pd.DataFrame(rows)

def revert_by_fixed(df):
    d = add_fixed_buckets(df); rows = []
    for (ch, b), g in d.groupby(['blockchain','size_fixed'], observed=True):
        if len(g) == 0: continue
        p, lo, hi = wilson(int(g['reverted'].sum()), len(g))
        rows.append({'chain': ch, 'bucket': b, 'n': len(g), 'revert_rate': p, 'ci_lo': lo, 'ci_hi': hi})
    return pd.DataFrame(rows)

def plot_revert_by_size(qtbl, df):
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    for ch, g in qtbl.groupby('chain'):
        g = g.set_index('size_q').reindex([f'Q{i}' for i in range(1, 6)])
        lo = (g['revert_rate']-g['ci_lo']).clip(lower=0).fillna(0); hi = (g['ci_hi']-g['revert_rate']).clip(lower=0).fillna(0)
        ax[0].errorbar(range(len(g)), g['revert_rate']*100, yerr=np.vstack([lo, hi])*100, marker='o', capsize=3, label=ch)
    ax[0].set_xticks(range(5)); ax[0].set_xticklabels([f'Q{i}' for i in range(1, 6)])
    ax[0].set_xlabel('per-chain size quintile'); ax[0].set_ylabel('revert rate (%)')
    ax[0].set_title('Revert rate by per-chain size quantile'); ax[0].legend()
    for ch, d in df.groupby('blockchain'):
        v = d[['size_usd','reverted']].dropna(); v = v[v['size_usd'] > 0]
        if len(v) < 50: continue
        lx = np.log10(v['size_usd']); bins = np.linspace(lx.min(), lx.max(), 25); idx = np.digitize(lx, bins)
        xs, ys = [], []
        for b in range(1, len(bins)):
            m = idx == b
            if m.sum() >= 20: xs.append(bins[b-1]); ys.append(v['reverted'][m].mean()*100)
        ax[1].plot(xs, ys, marker='.', label=ch)
    ax[1].set_xlabel('log10(size_usd)'); ax[1].set_ylabel('revert rate (%)')
    ax[1].set_title('Rolling revert rate vs size (boundary-free)'); ax[1].legend()
    plt.tight_layout(); plt.show()

T4q = revert_by_quantile(DF); print('=== per-chain quantile (primary) ==='); display(T4q)
T4f = revert_by_fixed(DF); print('=== fixed-$ buckets (secondary) ===')
display(T4f.pivot_table(index='bucket', columns='chain', values='revert_rate', observed=True))
plot_revert_by_size(T4q, DF)

## 3. Cap vs economics (reward / both P&L concepts)
Per-chain median reward, gross P&L (execution-only) and net P&L (incl reward/penalty) vs penalty cap.

In [ ]:
def cap_vs_economics(df):
    rps = reward_per_auction_solver(df)
    reward = rps.groupby('blockchain').agg(med_reward_usd=('reward_usd','median'),
        med_penalty_usd=('penalty_usd','median'), med_net_pnl_usd=('net_mechanism_pnl_usd','median'))
    other = df.groupby('blockchain').agg(penalty_cap_usd=('penalty_cap_usd','median'),
        med_gross_pnl_usd=('gross_execution_pnl_usd','median'), n_attempts=('order_uid','size'))
    return other.join(reward).sort_values('penalty_cap_usd')

def plot_cap_vs_economics(tbl):
    metrics = [('med_reward_usd','median reward (USD)'), ('med_gross_pnl_usd','median GROSS pnl (exec-only)'),
               ('med_net_pnl_usd','median NET pnl (incl reward/penalty)')]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    for (m, lab), a in zip(metrics, ax):
        if len(tbl) >= 2:
            a.scatter(tbl['penalty_cap_usd'], tbl[m], s=60)
            for ch, r in tbl.iterrows(): a.annotate(ch, (r['penalty_cap_usd'], r[m]))
        else: barpos(a, list(tbl.index), tbl[m])
        a.set_xlabel('penalty cap (USD)'); a.set_ylabel(lab); a.set_title(lab)
    plt.tight_layout(); plt.show()

T5 = cap_vs_economics(DF); display(T5); plot_cap_vs_economics(T5)

## 4. Cap vs markout (surplus delivered to users)
markout = net USD surplus delivered at execution (NOT post-trade adverse selection). Hypothesis: higher cap leads to less surplus passed through. ECDF per chain + median by chain.

In [ ]:
def markout_vs_cap(df):
    d = add_fixed_buckets(df[df['settled'] == True])
    by_chain = d.groupby('blockchain').agg(penalty_cap_usd=('penalty_cap_usd','median'),
        med_markout_bps=('markout_bps','median'), n=('order_uid','size')).sort_values('penalty_cap_usd')
    by_size = d.groupby(['blockchain','size_fixed'], observed=True)['markout_bps'].median().unstack(0)
    return by_chain, by_size, d

def plot_markout(by_chain, d):
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    barpos(ax[0], list(by_chain.index), by_chain['med_markout_bps'], color='#9bbb59')
    ax[0].axhline(0, color='k', lw=.8); ax[0].set_ylabel('median markout (bps)')
    ax[0].set_title('Surplus to users by chain')
    for ch, g in d.groupby('blockchain'): ecdf(ax[1], g['markout_bps'].clip(-200, 200), ch)
    ax[1].set_xlabel('markout (bps, clipped ±200)'); ax[1].set_ylabel('ECDF'); ax[1].set_title('Markout distribution'); ax[1].legend()
    plt.tight_layout(); plt.show()

T6c, T6s, T6d = markout_vs_cap(DF); display(T6c); display(T6s); plot_markout(T6c, T6d)

## 5. Three outcomes vs cap
Solver gross P&L, user markout, time-to-happy-moo vs cap on one panel. Hypothesis: higher cap leads to worse markout + longer settle. (TTHM coverage sparse for the limit-order population.)

In [ ]:
def three_outcomes(df):
    st = df[df['settled'] == True]
    base = df.groupby('blockchain').agg(penalty_cap_usd=('penalty_cap_usd','median'),
        med_gross_pnl_usd=('gross_execution_pnl_usd','median'))
    return base.join([st.groupby('blockchain')['markout_bps'].median().rename('med_markout_bps'),
        st.groupby('blockchain')['seconds_to_settle'].median().rename('med_tthm_s')]).sort_values('penalty_cap_usd')

def plot_three(tbl):
    metrics = [('med_gross_pnl_usd','solver gross P&L (USD)'), ('med_markout_bps','markout (bps, user)'),
               ('med_tthm_s','time-to-happy-moo (s)')]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    for (m, lab), a in zip(metrics, ax):
        if len(tbl) >= 2:
            a.scatter(tbl['penalty_cap_usd'], tbl[m], s=60)
            for ch, r in tbl.iterrows(): a.annotate(ch, (r['penalty_cap_usd'], r[m]))
        else: barpos(a, list(tbl.index), tbl[m])
        a.set_xlabel('penalty cap (USD)'); a.set_ylabel(lab); a.set_title(lab)
    plt.tight_layout(); plt.show()

T7 = three_outcomes(DF); display(T7); plot_three(T7)

## 6. Distribution of capped vs uncapped penalties, per chain.
On reverts: how often the cap binds and how much penalty it forgives. 

In [ ]:
def cap_binding(df):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted']]; rows = []
    for ch, d in rev.groupby('blockchain'):
        unc = d['penalty_uncapped_usd']; cap = d['penalty_usd']; w = unc.quantile(0.999)
        rows.append({'chain': ch, 'reverts': len(d), 'cap_binds_rate': d['cap_binds'].mean(),
            'med_penalty_on_revert_usd': cap.median(), 'capped_sum_usd': cap.sum(), 'uncapped_sum_usd': unc.sum(),
            'forgiven_sum_usd': (unc-cap).sum(), 'forgiven_sum_winsor_usd': (np.minimum(unc, w)-cap).sum()})
    return pd.DataFrame(rows).set_index('chain')

def plot_cap_binding(df, tbl):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted'] & (rev['penalty_uncapped_usd'] > 0)]
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for ch, d in rev.groupby('blockchain'):
        ecdf(ax[0], np.log10(d['penalty_uncapped_usd'].clip(lower=1e-6)), f'{ch} uncapped')
        ax[0].axvline(np.log10(d['penalty_cap_usd'].median()), ls='--', alpha=.6)
    ax[0].set_xlabel('log10 uncapped penalty (USD)'); ax[0].set_ylabel('ECDF')
    ax[0].set_title('Uncapped penalty vs cap (dashed)'); ax[0].legend(fontsize=8)
    barpos(ax[1], list(tbl.index), tbl['cap_binds_rate']*100, color='#c0504d')
    ax[1].set_ylabel('% reverts where cap binds'); ax[1].set_title('Cap-binding rate')
    plt.tight_layout(); plt.show()

T8 = cap_binding(DF); display(T8); plot_cap_binding(DF, T8)

## 7. Variable-rate penalty counterfactual
Recompute penalties had the cap been a variable rate (% of size): `min(uncapped, r·size)`: a variable *cap*, bidding held constant. 

In [ ]:
def variable_rate(df, grid=(1, 2, 5, 10, 20, 50)):
    rev = reward_per_auction_solver(df); rev = rev[rev['reverted']].copy()
    rev = rev[rev['volume_usd'].notna() & (rev['volume_usd'] > 0)]
    out = {}
    for ch, d in rev.groupby('blockchain'):
        cur = d['penalty_usd'].sum(); gate = d['volume_usd'].quantile(0.999)
        recs = [{'rate': 'current flat cap', 'total_usd': cur, 'pct_vs_current': 0.0}]
        for r in grid:
            tot = np.minimum(d['penalty_uncapped_usd'], r/1e4*d['volume_usd']).sum()
            recs.append({'rate': f'{r} bps', 'total_usd': tot, 'pct_vs_current': 100*(tot-cur)/cur if cur else np.nan})
        lo, hi = 0.0, 1000.0                                       # revenue-neutral bps via bisection on capped formula
        for _ in range(40):
            mid = (lo+hi)/2; tot = np.minimum(d['penalty_uncapped_usd'], mid/1e4*d['volume_usd']).sum()
            lo, hi = (mid, hi) if tot < cur else (lo, mid)
        out[ch] = (pd.DataFrame(recs).set_index('rate'), (lo+hi)/2, cur, d.loc[d['volume_usd'] < gate, 'penalty_usd'].sum())
    return out

def plot_variable(res):
    fig, ax = plt.subplots(figsize=(10, 5))
    for ch, (tbl, rn, cur, _) in res.items():
        ax.plot(tbl.index, tbl['total_usd'], marker='o', label=f'{ch} (rev-neutral ≈ {rn:.1f} bps)')
    ax.set_ylabel('total penalty (USD)'); ax.set_xlabel('regime'); ax.tick_params(axis='x', rotation=30)
    ax.set_title('Flat cap vs variable-rate penalty (bidding held constant)'); ax.legend()
    plt.tight_layout(); plt.show()

T9 = variable_rate(DF)
for ch, (tbl, rn, cur, gated) in T9.items():
    print(f'\n=== {ch} === revenue-neutral ≈ {rn:.2f} bps | current total ${cur:,.0f} | gated current ${gated:,.0f}'); display(tbl)
plot_variable(T9)

## 8. Sit-out (non-economic penalty) simulation
Approximate suspending a reverting solver for X blocks (via `block_deadline`): tally forgone reward, avoided penalties, removed attempts, coverage-at-risk. 

In [ ]:
def sit_out(df, X_blocks=300):
    res = []
    base = reward_per_auction_solver(df)
    bd_map = df.groupby(['blockchain','auction_id','solver'])['block_deadline'].first().reset_index()
    base = base.merge(bd_map, on=['blockchain','auction_id','solver'], how='left')
    for (ch, solver), d in base.sort_values('block_deadline').groupby(['blockchain','solver']):
        d = d.reset_index(drop=True)
        bd = d['block_deadline'].to_numpy(float); rev = d['reverted'].to_numpy()
        rew = d['reward_usd'].fillna(0).to_numpy(); pen = d['penalty_usd'].fillna(0).to_numpy()
        oc = d['order_count'].fillna(1).to_numpy()              
        forg = avoid = 0; removed_orders = covg_orders = 0; ban_until = -np.inf
        for i in range(len(d)):
            if rev[i] and bd[i] > ban_until:                      # non-overlapping bans
                w = (bd > bd[i]) & (bd <= bd[i] + X_blocks)
                forg += rew[w].sum(); avoid += pen[w].sum()
                removed_orders += int(oc[w].sum())                
                covg_orders += int(oc[w & ~rev].sum())            
                ban_until = bd[i] + X_blocks
        res.append({'chain': ch, 'solver': d['solver_name'].iloc[0] if d['solver_name'].notna().any() else solver,
            'reverts': int(rev.sum()), 'forgone_reward_usd': forg, 'avoided_penalty_usd': avoid,
            'removed_orders': removed_orders, 'coverage_at_risk_orders_ub': covg_orders})
    out = pd.DataFrame(res); out['net_cost_usd'] = out['forgone_reward_usd'] - out['avoided_penalty_usd']
    return out.sort_values('net_cost_usd', ascending=False)

def plot_sit_out(tbl, X, top=15):
    t = tbl.head(top); labels = [str(s) for s in t['solver']][::-1]; y = np.arange(len(labels)); w = 0.4
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.barh(y+w/2, t['forgone_reward_usd'][::-1], w, label='forgone reward', color='#8064a2')
    ax.barh(y-w/2, -t['avoided_penalty_usd'][::-1], w, label='avoided penalty', color='#4f81bd')
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.axvline(0, color='k', lw=.8)
    ax.set_xlabel(f'USD under {X}-block sit-out'); ax.set_title(f'Sit-out cost by solver (top {top} by net cost)'); ax.legend()
    plt.tight_layout(); plt.show()

X_BLOCKS = 300
T10 = sit_out(DF, X_BLOCKS); display(T10.head(15)); plot_sit_out(T10, X_BLOCKS)

## 9. Within-solver paired comparison (cross-chain)
Same solver on different-cap chains, controls for solver skill/infra/flow, cross-chain confounder. Needs ≥2 chains.

In [ ]:
def within_solver(df):
    g = df.groupby(['solver_name','blockchain']).agg(n=('order_uid','size'), revert_rate=('reverted','mean'),
        med_gross_pnl_usd=('gross_execution_pnl_usd','median'), med_net_pnl_usd=('net_mechanism_pnl_usd','median'),
        med_markout_bps=('markout_bps','median'), penalty_cap_usd=('penalty_cap_usd','median')).reset_index()
    multi = g.groupby('solver_name')['blockchain'].transform('nunique') > 1
    return g[multi.values].sort_values(['solver_name','penalty_cap_usd'])

def plot_within_solver(tbl):
    if tbl.empty or tbl['blockchain'].nunique() < 2:
        print('Need solvers active on ≥2 chains.'); return
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    for sv, g in tbl.groupby('solver_name'):
        g = g.sort_values('penalty_cap_usd')
        ax[0].plot(g['penalty_cap_usd'], g['revert_rate']*100, marker='o', label=sv)
        ax[1].plot(g['penalty_cap_usd'], g['med_gross_pnl_usd'], marker='o', label=sv)
    ax[0].set_ylabel('revert rate (%)'); ax[1].set_ylabel('median gross P&L (USD)')
    for a in ax: a.set_xlabel('penalty cap (USD)'); a.legend(fontsize=7)
    ax[0].set_title('Within-solver revert rate vs cap'); ax[1].set_title('Within-solver P&L vs cap')
    plt.tight_layout(); plt.show()

T11 = within_solver(DF); display(T11.head(20)); plot_within_solver(T11)

## 10. Headline table
One compact per-chain summary for export.

In [ ]:
def headline(df):
    cov = coverage_table(df); rv = revert_by_chain(df); cb = cap_binding(df)
    st = df[df['settled'] == True].groupby('blockchain')
    return pd.DataFrame({'attempts': cov['attempts'], 'orders': cov['orders'], 'solvers': cov['solvers'],
        'revert_rate': rv['revert_rate'], 'ci_lo': rv['ci_lo'], 'ci_hi': rv['ci_hi'],
        'penalty_cap_usd': cov['penalty_cap_usd'], 'cap_binds_rate': cb['cap_binds_rate'],
        'median_size_usd': cov['median_size_usd'], 'med_markout_bps': st['markout_bps'].median(),
        'med_tthm_s': st['seconds_to_settle'].median(), 'solver_hhi': cov['solver_hhi']}).round(3)

display(headline(DF))
print('\nfunnel:', {k: f'{v:,}' for k, v in FUNNEL.items()})
if not MULTI: print('Single chain')